### Data Ingestion

In [1]:
### Document Structure

from langchain_core.documents import Document                                       

In [2]:
doc = Document(
    page_content = "This is the main text content I am using to create RAG",
    metadata = {
        "source" : "example.txt",
        "pages" : 1,
        "author" : "DAK",
        "date_created" : "2026-02-25"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'DAK', 'date_created': '2026-02-25'}, page_content='This is the main text content I am using to create RAG')

In [3]:
## Create a simple txt file

import os
os.makedirs("/data/text_files", exist_ok = True)

In [4]:
sample_texts = {"/data/text_files/python_intro.txt" : """Python Programming Introduction
                
Python is a high-level, interpreted programming language known for its simplicity, readability, and versatility. 
Created by Guido van Rossum and first released in 1991, Python was designed with an emphasis on clear syntax and code readability. 
Its clean structure allows developers to express complex logic using fewer lines of code compared to many other programming languages. 
Because of this, Python is widely recommended as a first programming language for beginners while also being powerful enough for advanced software development.""",

"/data/text_files/machine_learning.txt" : """Machine Learning Basics 
Machine learning is a subset of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions without being explicitly programmed.
Instead of following fixed rules, machine learning algorithms improve their performance by training on historical data. 
It primarily includes supervised learning, unsupervised learning, and reinforcement learning techniques. 
Machine learning is widely used in applications such as recommendation systems, fraud detection, image recognition, and predictive analytics.
"""
}

for filepath, content in sample_texts.items() :
    with open(filepath, 'w', encoding = "utf-8") as f:
        f.write(content)

print("✅ Sample text files created successfully!!!!")

✅ Sample text files created successfully!!!!


In [5]:
### TextLoader
from langchain_community.document_loaders import TextLoader

d:\DAK_AIML_Resume Projects\RAG Pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
loader = TextLoader("/data/text_files/python_intro.txt", encoding = "utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '/data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity, readability, and versatility. \nCreated by Guido van Rossum and first released in 1991, Python was designed with an emphasis on clear syntax and code readability. \nIts clean structure allows developers to express complex logic using fewer lines of code compared to many other programming languages. \nBecause of this, Python is widely recommended as a first programming language for beginners while also being powerful enough for advanced software development.')]


In [7]:
### Directory Loader

from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "data/text_files",
    glob = "**/*.txt", ## Pattern to match files
    loader_cls = TextLoader, ##Loader class to use
    loader_kwargs = {'encoding' : 'utf-8'},
    show_progress = False
)
documents = dir_loader.load()
documents

[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics \nMachine learning is a subset of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions without being explicitly programmed.\nInstead of following fixed rules, machine learning algorithms improve their performance by training on historical data. \nIt primarily includes supervised learning, unsupervised learning, and reinforcement learning techniques. \nMachine learning is widely used in applications such as recommendation systems, fraud detection, image recognition, and predictive analytics.\n'),
 Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity, readability, and versatility. \nCreated by Guido van Rossum and first released in 1991, Python was designed with an emphasis on clear

In [8]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader

## load all the PDF files from the directory
dir_loader=DirectoryLoader(
    "data/pdf files",
    glob = "**/*.pdf", ## Pattern to match files
    loader_cls = PyMuPDFLoader, ##Loader class to use
    show_progress = False
)
pdf_documents = []
for doc in dir_loader.lazy_load():
    pdf_documents.append(doc)


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

all_chunks = []

for pdf_path in Path("data/pdf files").rglob("*.pdf"):
    loader = PyMuPDFLoader(str(pdf_path))
    docs = loader.load()
    chunks = text_splitter.split_documents(docs)
    all_chunks.extend(chunks)

In [10]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### Embedding and VectorStoreDB

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [12]:
from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer

In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name : str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name : HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self) :
        """Load the SentenceTransformer model"""
        try :
            print(f"Loading embedding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension : {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise

    def generate_embeddings(self, texts : List[str]) -> np.ndarray :
        """ 
        Generate embeddings for a list of texts
        
        Args :
            texts : List of text strings to embed
            
        Returns :
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model :
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape : {embeddings.shape}")
        return embeddings
    
## Initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager
    



Loading embedding model : all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension : 384


### VectorStore

In [14]:
class VectorStore :
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name : str = "pdf_documents", persist_directory : str = "../data/vector_store"):
        """ 
        Initialize the vector Store
        
        Args :
            collection name : Name of the ChromaDB collection
            persist_directory : Directory to persist the vector store"""
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self) :
        """Initialize ChromaDB client and collection"""
        try :
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description" : "PDF document embeddings for RAG"}
            )

            print(f"Vector Store initialized. Collection : {self.collection_name}")
            print(f"Existing documents in collections : {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise

    def add_documents(self, documents : List[Any], embeddings : np.ndarray) :
        """ 
        Add documents and their embeddings to the vector store
        
        Args :
            documents : List of langchain documents
            embeddings : Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings) :
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store... ")


        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)) :
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try :
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection : {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store : {e}")
            raise

vectorstore = VectorStore()
vectorstore


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector Store initialized. Collection : pdf_documents
Existing documents in collections : 0


In [15]:
### Chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200, separators = ["\n\n", "\n", " ",""])

pdf_chunks = text_splitter.split_documents(pdf_documents)

print(f"Original documents : {len(pdf_documents)}")
print(f"Chunks created : {len(pdf_chunks)}")

pdf_chunks[0].page_content[:500]
pdf_chunks[0].metadata

Original documents : 9
Chunks created : 33


{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-02-25T06:13:53+00:00',
 'source': 'data\\pdf files\\attention.pdf',
 'file_path': 'data\\pdf files\\attention.pdf',
 'total_pages': 2,
 'format': 'PDF 1.4',
 'title': '(anonymous)',
 'author': '(anonymous)',
 'subject': '(unspecified)',
 'keywords': '',
 'moddate': '2026-02-25T06:13:53+00:00',
 'trapped': '',
 'modDate': "D:20260225061353+00'00'",
 'creationDate': "D:20260225061353+00'00'",
 'page': 0}

In [16]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-02-25T06:13:53+00:00', 'source': 'data\\pdf files\\proposal.pdf', 'file_path': 'data\\pdf files\\proposal.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-02-25T06:13:53+00:00', 'trapped': '', 'modDate': "D:20260225061353+00'00'", 'creationDate': "D:20260225061353+00'00'", 'page': 0}, page_content="Research Proposal: Efficient Long-Context\nTransformers for Document Understanding\n1. Introduction and Motivation\nLarge language models (LLMs) based on the Transformer architecture have achieved remarkable\nresults across a wide range of natural language processing tasks. However, the standard self-attention\nmechanism scales quadratically with sequence length, making it computationally prohibitive to process\nlong documents — legal contracts, scientific papers, med

In [17]:
### Convert the text to embeddings

texts = [doc.page_content for doc in chunks]
texts

["Research Proposal: Efficient Long-Context\nTransformers for Document Understanding\n1. Introduction and Motivation\nLarge language models (LLMs) based on the Transformer architecture have achieved remarkable\nresults across a wide range of natural language processing tasks. However, the standard self-attention\nmechanism scales quadratically with sequence length, making it computationally prohibitive to process\nlong documents — legal contracts, scientific papers, medical records, financial filings — in their\nentirety.\nCurrent practical approaches include truncating documents to the model's context window (typically\n512–4096 tokens), chunking documents and processing segments independently, or\nretrieval-augmented generation (RAG) which retrieves relevant passages before generation. Each of\nthese has significant limitations: truncation loses information, chunking breaks long-range\ndependencies, and RAG requires a separate retrieval pipeline and can miss relevant context.",
 'the

In [18]:
### Convert the text to embeddings

texts = [doc.page_content for doc in chunks]

## Generate the embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorstore.add_documents(chunks, embeddings)



Generating embeddings for 9 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.05it/s]
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Generated embeddings with shape : (9, 384)
Adding 9 documents to vector store... 
Successfully added 9 documents to vector store
Total documents in collection : 9


## Retriever Pipeline from VectorStore

In [19]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """ 
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """ 
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in Vector Store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]   # ✅ Fix 1
                distances = results['distances'][0]   # ✅ Fix 1
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):  # ✅ Fix 2
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - float(distance)

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,   # ✅ Fix 2
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No document found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

rag_retriever



In [20]:
rag_retriever.retrieve("What is attention is all you need 2")

Retrieving documents for query: 'What is attention is all you need 2'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 108.71it/s]
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Generated embeddings with shape : (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_9662a01f_3',
  'content': 'guidelines for practitioners deploying these systems.\n3. Proposed Methodology\nFor the hierarchical attention model, we will build on the Longformer and BigBird architectures but\nintroduce a learned routing mechanism between levels. Sentence-level representations will be\ncomputed using a standard Transformer with full local attention; paragraph-level representations will be',
  'metadata': {'author': '(anonymous)',
   'content_length': 375,
   'creationDate': "D:20260225061353+00'00'",
   'creationdate': '2026-02-25T06:13:53+00:00',
   'creator': '(unspecified)',
   'doc_index': 3,
   'file_path': 'data\\pdf files\\proposal.pdf',
   'format': 'PDF 1.4',
   'keywords': '',
   'modDate': "D:20260225061353+00'00'",
   'moddate': '2026-02-25T06:13:53+00:00',
   'page': 0,
   'producer': 'ReportLab PDF Library - (opensource)',
   'source': 'data\\pdf files\\proposal.pdf',
   'subject': '(unspecified)',
   'title': '(anonymous)',
   'total_pages': 3

### Integration VectorDB Context Pipeline with LLM Output

In [21]:
### Simple RAG Pipeline with GROQ LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

## 2. Simple RAG function function :  retrieve context + generate response
def rag_simple(query, retriever, llm, top_k = 3) :

    ## retrieve the context
    results = retriever.retrieve(query, top_k = top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context :
        return "No relevant context found to answer the question."
    
    ## Generate the answer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
        Context :
        {context}

        Question : {query}

        Answer : """
    
    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content




In [22]:
answer = rag_simple("What is attention mechanism?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 85.23it/s]

Generated embeddings with shape : (1, 384)
Retrieved 1 documents (after filtering)


The attention mechanism is a technique used in deep learning models, such as Transformers, to focus on specific parts of the input data when generating output. It allows the model to weigh the importance of different input elements and allocate more attention to the most relevant ones.


### Enhanced RAG Pipeline Feature

In [23]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(query, retriever, llm, top_k = 5, min_score = 0.2, return_context = False) :
    """ 
    RAG Pipeline with extra features :
    - Returns answer, sources, confidence score and optionally full context.
    """

    results = retriever.retrieve(query, top_k = top_k, score_threshold = min_score)
    if not results :
        return {'answer' : 'No relevant context found.', 'sources' : [], 'confidence' :0.0, 'context' : ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page' : doc['metadata'].get('page', 'unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:300]+ '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext : \n {context}\n\n Question : {query} \n\nAnswer :""" 
    response = llm.invoke([prompt.format(context = context, query = query)])

    output = {
        'answer' : response.content,
        'sources' : sources,
        'confidence' : confidence
    }
    if return_context :
        output['context'] = context
    return output

# Example usage :
result = rag_advanced("what is attention mechanism?", rag_retriever, llm, top_k = 3, min_score = 0.1, return_context = True)
print("Answer : ", result['answer'])
print("Sources : ", result['sources'])
print("Confidence : ", result['confidence'])
print("Context Preview : ", result['context'][:300])

Retrieving documents for query: 'what is attention mechanism?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 140.98it/s]

Generated embeddings with shape : (1, 384)
Retrieved 0 documents (after filtering)
Answer :  No relevant context found.
Sources :  []
Confidence :  0.0
Context Preview :  


In [24]:
# -- Advanced RAG Pipeline : Streaming, Citations, History, Summarization ---

from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False,
    ) -> Dict[str, Any]:

        # Retrieve Relevant Documents
        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        sources = []
        context = ""

        if not results:
            answer = "No relevant context found."
        else:
            context = "\n\n".join([doc["content"] for doc in results])

            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file",
                        doc["metadata"].get("source", "unknown"),
                    ),
                    "page": doc["metadata"].get("page", "unknown"),
                    "score": doc.get("similarity_score", 0.0),
                    "preview": doc["content"][:120] + "...",
                }
                for doc in results
            ]

            # Prompt construction
            prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question:
{question}

Answer:
"""

            # Optional streaming simulation
            if stream:
                print("Streaming answer:\n")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i + 80], end="", flush=True)
                    time.sleep(0.05)
                print("\n")

            # Proper ChatGroq invocation (expects messages list)
            response = self.llm.invoke(
                [{"role": "user", "content": prompt}]
            )

            answer = response.content

        # Add citations
        if sources:
            citations = [
                f"[{i+1}] {src['source']} (page {src['page']})"
                for i, src in enumerate(sources)
            ]
            answer_with_citations = (
                answer + "\n\nCitations:\n" + "\n".join(citations)
            )
        else:
            answer_with_citations = answer

        # Optional summarization
        summary = None
        if summarize and answer:
            summary_prompt = f"""Summarize the following answer in 2 sentences:

{answer}
"""
            summary_resp = self.llm.invoke(
                [{"role": "user", "content": summary_prompt}]
            )
            summary = summary_resp.content

        # Store Query History
        self.history.append(
            {
                "question": question,
                "answer": answer,
                "sources": sources,
                "summary": summary,
            }
        )

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history,
        }


# Example usage
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)

result = adv_rag.query(
    "What is attention mechanism in deep learning?",
    top_k=3,
    min_score=0.2,
    stream=True,
    summarize=True,
)

print("\nFinal Answer:\n", result["answer"])
print("\nSummary:\n", result["summary"])
print("\nLast History Entry:\n", result["history"][-1])

Retrieving documents for query: 'What is attention mechanism in deep learning?'
Top K: 3, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<?, ?it/s]

Generated embeddings with shape : (1, 384)
Retrieved 0 documents (after filtering)

Final Answer:
 No relevant context found.

Summary:
 There is no information provided to summarize.

Last History Entry:
 {'question': 'What is attention mechanism in deep learning?', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'There is no information provided to summarize.'}
